# MVP of Kompressor for evaluation against Scientific Data

If you have never used Flax and Jax before it is worth going through this resource till you feel comfortable: https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial2/Introduction_to_JAX.html

Import required packages

In [ ]:
import os
import time
import typing
from collections import defaultdict
from pathlib import Path
from typing import List, Optional, Tuple, Union

import flax.linen as nn
import jax
import jax.numpy as jnp
import kompressor as kom
import matplotlib.pyplot as plt
import mrcfile
import numpy as np
import optax
import orbax.checkpoint as ocp
import sklearn.model_selection as model_selection
import torch
import torchvision
import torchvision.transforms as transforms
import wandb
from jax.tree_util import tree_map
from torch.utils import data
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Types

In [ ]:
Paths = str
Frame = int
DataPathFrame = Tuple[Paths, Frame]

# Constants

In [ ]:
UINT16_MAX_VALUE: float = 65536.0

# Create the config handler

In [ ]:
config = {}

### Select the number of levels (resolutions) to train the model over.

A level of will train to reduce the image by a skip size of $2^{level}$ so a level of $2$ ($2^2=4$) will result in an image a quarter the size. 


In [ ]:
config["level"] = 1

### Select neighbourhood size for prediction model

In [ ]:
config["neighbourhood"] = 2

# At padding=0 use neighbours [A-D] to predict values [0-4]

# AB    A2B
# CD    041
#       C3D

# At padding=1 use neighbours [A-P] to predict values [0-4]

# ABCD    A.B.C.D
# EFGH    .......
# IJKL    E.F2G.H
# MNOP    ..041..
#         I.J3K.L
#         .......
#         M.N.O.P

# Define Convolutional Super Resolution Model

The model will take in [Batch, 259,259,1] and needs to output [Batch, 256,256,5,1,] 

The 5 represents the 5 maps: left, right, up, down and center. The maps are later merged to create lrmap, udmap and cmap. 

This is a very simple model so performance may vary, it is worth exploring and adapting other super resolution models: https://github.com/isaaccorley/jax-enhance has a collection of architectures that would be changed. 

It's important to note that this model does not need to upsampling. As we are trying to predict the high resolution maps from the lowers image. 

Other hyperamaters to change: 

* Depth: How many layers
* Width: How wide the layers are
* Skip Connections: ResNets are a good example https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial5/Inception_ResNet_DenseNet.html
* Activation Function: https://flax.readthedocs.io/en/latest/api_reference/flax.linen/activation_functions.html#module-flax.linen.activation
* Batch Norm: https://flax.readthedocs.io/en/latest/guides/training_techniques/batch_norm.html

In [ ]:
class unetConv(nn.Module): 
    padding: int
    out_dim: int

    #Conv Parameters
    kernel_size: tuple = (3, 3)
    strides: int = 1

    use_batchnorm: bool = False

    # BatchNorm Parameters
    use_running_average: bool = False
    momentum: float = 0.9
    epsilon: float = 1e-5
    dtype: jnp.dtype = jnp.float32
    
    @nn.compact
    def __call__(self, x):
        if self.use_batchnorm:
            x = nn.Conv(features = self.out_dim, kernel_size = self.kernel_size, strides = self.strides, padding = 'SAME')(x) 
            x = nn.BatchNorm(use_running_average = self.use_running_average, momentum = self.momentum, epsilon = self.epsilon, dtype = self.dtype)(x) 
            x = nn.relu(x)
            x = nn.Conv(features = self.out_size, kernel_size = self.kernel_size, strides = self.strides, padding= 'SAME')(x)
            x = nn.BatchNorm(use_running_average = self.use_running_average, momentum = self.momentum, epsilon = self.epsilon, dtype = self.dtype)(x)
            x = nn.relu(x)
            return x
        else:
            x = nn.Conv(features = self.out_dim, kernel_size = self.kernel_size, strides = self.strides, padding = 'SAME')(x)
            x = nn.relu(x)
            x = nn.Conv(features = self.out_dim, kernel_size = self.kernel_size, strides = self.strides, padding = 'SAME')(x) 
            x = nn.relu(x)
            return x


class Upsample(nn.Module):  
    out_dim: int
    is_deconv: bool
    odd_even_height: int
    odd_even_width: int
    
    # ConvTranspose Parameters
    kernel_size: int = 2
    strides: int = 2

    @nn.compact
    def forward(self, inputs1, inputs2):
        
        if self.is_deconv:
            outputs2 = nn.ConvTranspose(features = self.out_dim, kernel_size = (self.kernel_size + self.odd_even_height, self.kernel_size + self.odd_even_width), strides = self.strides)(inputs2)
        else:
            outputs2 = nn.UpsamplingBilinear2d(scale_factor=2) (inputs2)

        offset = outputs2.size()[2] - inputs1.size()[2]

        padding = 2 * [offset // 2, offset // 2]

        outputs1 = jnp.pad(inputs1, padding)

        return unetConv(features = self.out_dim, is_batchnorm = False) (jnp.concatenate([outputs1, outputs2], 1))

class unet(nn.Module):
    feature_scale: int =4 
    n_classes: int = 21 
    is_deconv: bool = True  
    use_batchnorm: bool = True
    kernel_size: int = 2
    patches: int = 5
    channels: int = 1

    @nn.compact
    def __call__(self, x):

        is_deconv = self.is_deconv
        use_batchnorm = self.use_batchnorm
        feature_scale = self.feature_scale

        filters = [64, 128, 256, 512, 1024]
        filters = [int(x / feature_scale) for x in filters]

        # downsampling

        conv1 = unetConv(filters[0], use_batchnorm)(x)
        maxpool1 = nn.MaxPool2d(kernel_size = self.kernel_size)(conv1)

        conv2 = unetConv(filters[1], use_batchnorm)(maxpool1)
        maxpool2 = nn.MaxPool2d(kernel_size = self.kernel_size)(conv2)

        conv3 = unetConv(filters[2], use_batchnorm)(maxpool2)
        maxpool3 = nn.MaxPool2d(kernel_size = self.kernel_size)(conv3)

        conv4 = unetConv(filters[3], use_batchnorm)(maxpool3)
        maxpool4 = nn.MaxPool2d(kernel_size = self.kernel_size)(conv4)

        center = unetConv(filters[4], use_batchnorm)(maxpool4)

        # upsampling
        up4 = Upsample(filters[3], is_deconv = is_deconv, odd_even_height = conv3.shape[1], odd_even_width = conv3.shape[2])(conv4, center)
        up3 = Upsample(filters[2], is_deconv = is_deconv, odd_even_height = conv2.shape[1], odd_even_width = conv2.shape[2])(conv3, up4)
        up2 =  Upsample(filters[1], is_deconv = is_deconv, odd_even_height = conv1.shape[1], odd_even_width = conv1.shape[2])(conv2, up3)
        up1 = Upsample(filters[0], is_deconv = is_deconv, odd_even_height = x.shape[1], odd_even_width = x.shape[2])(conv1, up2)
        
        if self.use_batchnorm:
            final = nn.Conv(features = self.n_classes, kernel_size = (3,3), padding = "VALID")(up1) 
            final = nn.BatchNorm(use_running_average = False, momentum = 0.9, epsilon = 1e-5, dtype = jnp.float32)(final) 
            final = nn.relu(final)
            final = nn.Conv(features = self.n_classes, kernel_size = (2,2), padding = "VALID")(final)
            final = nn.BatchNorm(use_running_average = False, momentum = 0.9, epsilon = 1e-5, dtype = jnp.float32)(final)
            final = nn.relu(final)
        else:
            final = nn.Conv(features = self.n_classes, kernel_size = (3,3), padding = "VALID")(up1)
            final = nn.relu(final)
            final = nn.Conv(features = self.n_classes, kernel_size = (2,2), padding = "VALID")(final) 
            final = nn.relu(final)

        batch, height, width = final.shape[0:3]
        return final.reshape(batch, height, width, self.patches, self.channels)

        return final

In [ ]:
a = unet(config["neighbourhood"])
print(
    a.tabulate(
        jax.random.key(0), jnp.ones((1, 259, 259, 1)), console_kwargs={"width": 125}
    )
)

### A Kompressor predictor function for a given network and parameter set that can be passed to encode/decode

In [ ]:
def RegressionPredictor(model, params):
    # Regression predictor function applies convolutional MLP network
    @jax.jit
    def predictions_fn(lowres):
        # lowres.shape == (B, H, W, C)

        # Get predictions for neighbourhoods
        predictions = model.apply(params, jnp.float32(lowres) / UINT16_MAX_VALUE)
        # Convert predictions to uint16
        predictions = jnp.floor(jnp.clip(predictions, 0, 1) * UINT16_MAX_VALUE).astype(
            lowres.dtype
        )
        # predictions.shape == (B, H, W, P, C)
        # where P = 5, the number of values that need to be predicted for each neighbourhood
        # Extract the maps from the predictions
        maps = kom.image.maps_from_predictions(predictions)
        # lrmap, udmap, cmap = maps
        return maps

    return predictions_fn

# Kompressor

Kompressor is designed in the style of Scikit-learn type functions, therefore it was encode, decode functions as well as fit

In [ ]:
class Kompressor:
    def __init__(self, encode_fn, decode_fn, padding):
        self.encode_fn, self.decode_fn = encode_fn, decode_fn
        self.padding = padding

    def _predictions_fn(self):
        raise NotImplementedError()

    def encode(self, highres, levels=1, chunk=None, progress_fn=None, debug=False):
        assert levels > 0

        predictions_fn = self._predictions_fn()

        maps = list()
        for level in range(levels):
            if chunk is None:
                lowres, maps_dims = kom.image.encode(
                    predictions_fn, self.encode_fn, highres, padding=self.padding
                )
            else:
                lowres, maps_dims = kom.image.encode_chunks(
                    predictions_fn,
                    self.encode_fn,
                    highres,
                    padding=self.padding,
                    chunk=chunk,
                    progress_fn=progress_fn,
                )

            if debug:
                maps.append((lowres, maps_dims, highres))
            else:
                maps.append(maps_dims)

            highres = lowres

        return lowres, maps

    def decode(self, lowres, maps, chunk=None, progress_fn=None, debug=False):
        assert len(maps) > 0

        predictions_fn = self._predictions_fn()

        for maps_dims in reversed(maps):
            if debug:
                _, maps_dims, _ = maps_dims

            if chunk is None:
                highres = kom.image.decode(
                    predictions_fn,
                    self.decode_fn,
                    lowres,
                    maps_dims,
                    padding=self.padding,
                )
            else:
                highres = kom.image.decode_chunks(
                    predictions_fn,
                    self.decode_fn,
                    lowres,
                    maps_dims,
                    padding=self.padding,
                    chunk=chunk,
                    progress_fn=progress_fn,
                )

            lowres = highres

        return highres

In [ ]:
class FlaxKompressor(Kompressor):
    def __init__(
        self, encode_fn, decode_fn, padding, model_fn, predictions_fn
    ):
        super().__init__(encode_fn=encode_fn, decode_fn=decode_fn, padding=padding)
        self.model_fn = model_fn
        self.__predictions_fn = predictions_fn
        self.params = self.avg_params = None
        self.local_devices = jax.local_devices()
        self.opt_state = None

    def _predictions_fn(self):
        return self.__predictions_fn(self.model_fn, self.avg_params)

    def init(self, ds_train, seed=None):
        """Initialize the model"""
        ds_train = iter(ds_train)
        if self.avg_params is None:
            self.avg_params = self.model_fn.init(
                jax.random.PRNGKey(seed or np.random.randint(1e6)),
                next(ds_train)["lowres"],
            )

        return self

    def fit(
        self,
        ds_train,
        start_step=0,
        end_step=1,
        learning_rate=1e-5,
        checkpoint_manager=None,
        callbacks=None,
    ):
        """
        Train the model
        Args:
            ds_train: Train Dataset
            start_step: Start Epoch
            end_step: End Epoch
            learning_rate: Rate to learn
            checkpoint_manager: Manager of Saving model weights
            callbacks: Callbacks for logging during training

        Returns:
            self
        """
        callbacks = callbacks or list()

        assert 0 <= start_step < end_step

        params = self.avg_params
        opt = optax.adam(learning_rate)
        if self.opt_state:
            opt_state = self.opt_state
        else:
            opt_state = opt.init(params)

        @jax.jit
        def l2(params):
            return 0.5 * sum(
                jnp.sum(jnp.square(param))
                for param in jax.tree_util.tree_leaves(params)
            )

        @jax.jit
        def loss(params, batch):
            predictions = self.model_fn.apply(params, batch["lowres"])
            prediction_loss = jnp.mean(optax.l2_loss(predictions, batch["targets"]))
            return prediction_loss + (1e-6 * l2(params))

        @jax.jit
        def update(params, opt_state, batch):
            value, grads = jax.value_and_grad(loss)(params, batch)
            updates, opt_state = opt.update(grads, opt_state)
            new_params = optax.apply_updates(params, updates)
            return value, new_params, opt_state

        @jax.jit
        def ema_update(params, avg_params):
            return optax.incremental_update(params, avg_params, step_size=0.001)

        # Train/eval loop
        train_len = len(ds_train)
        for epoch in tqdm(range(start_step, end_step), desc="Epochs"):
            for iters, train_batch in enumerate(tqdm(ds_train)):
                for callback in callbacks:
                    callback.on_step_start(
                        step=(epoch * train_len) + iters, compressor=self
                    )

                # Update params
                loss, params, opt_state = update(params, opt_state, train_batch)
                self.avg_params = ema_update(params, self.avg_params)
                for callback in callbacks:
                    callback.on_step_end(
                        step=(epoch * train_len) + iters, loss=loss, compressor=self
                    )
            if checkpoint_manager:
                checkpoint = {
                    "model": {"params": self.avg_params, "opt_state": opt_state}
                }
                checkpoint_manager.save(
                    args=ocp.args.StandardSave(checkpoint), step=epoch
                )
        # Return self to enable chaining
        return self

## Metric data Collection

In [ ]:
class Callback:
    def on_step_start(self, *args, **kargs):
        pass

    def on_step_end(self, *args, **kargs):
        pass

In [ ]:
class MetricsCallback(Callback):
    def __init__(self, chunk, ds_train, ds_test=None, log_freq=1, levels=1):
        super().__init__()

        assert log_freq > 0
        self.log_freq = log_freq

        assert levels >= 0
        self.levels = levels

        self.chunk = chunk

        self.ds_train = ds_train
        self.ds_test = ds_test

    def on_step_end(self, step, loss, compressor, *args, **kargs):
        # Record Loss at every step
        wandb.log({"train/loss": loss}, step=step)
        if not ((step > 0) and (step % self.log_freq == 0)):
            return

        def log(dataset, mode):
            start = time.time()
            summaries = defaultdict(list)
            for highres in dataset:
                lowres, level_encoded_maps = compressor.encode(
                    highres, levels=self.levels, chunk=self.chunk, debug=True
                )

                for level, (
                    level_highres,
                    (level_encoded_maps, _),
                    level_lowres,
                ) in enumerate(level_encoded_maps):
                    level_highres_maps = kom.image.maps_from_highres(level_highres)

                    writer_path = os.path.join(
                        f"{mode}/level={level}/",
                        f'lowres={"x".join(map(str, level_lowres.shape[1:]))}',
                    )

                    for label in level_encoded_maps.keys() & level_highres_maps.keys():
                        summaries[f"{writer_path}/{label}/run_length"].extend(
                            list(
                                kom.image.metrics.mean_run_length(
                                    level_encoded_maps[label]
                                ).flatten()
                            )
                        )

            for key in summaries.keys():
                wandb.log(
                    {key: (jnp.array(summaries[key]).flatten().mean())}, step=step
                )
            end = time.time()
            return end - start

        if self.ds_train is not None:
            train_eval_time = log(self.ds_train, "train")
            wandb.log({"train/eval_time": train_eval_time}, step=step)

        if self.ds_test is not None:
            test_eval_time = log(self.ds_test, "test")
            wandb.log({"test/eval_time": test_eval_time}, step=step)

# MRCFILE handling

In [ ]:
def decode_mrc_data_path(file,data_paths) -> np.array:
    """
    Decode the MRC data path returning the slice of the data speficifed.

    Args:
        data_paths: A List containing the [Path, Frame] tuples

    Returns:
        A Numpy array containing the decoded data along with an additional axis to create Height, Width, Channel.

    Examples:
        If we have an Data path array of ("/tmp/0.mrc",0) with shape (5,5) this will
        return the numpy array with the shape (5,5,1)
    """
    mrc_file = file
    frame_index = np.argmax(mrc_file.data.shape)
    if len(mrc_file.data.shape) == 3:
        if frame_index == 0:
            data = np.asarray(mrc_file.data[data_paths[1], ...])
        elif frame_index == 1:
            data = np.asarray(mrc_file.data[:, data_paths[1], ...])
        elif frame_index == 2:
            data = np.asarray(mrc_file.data[..., data_paths[1]])
    return np.expand_dims(data, axis=data.ndim)


def get_data_paths_and_frames(files: List[str]) -> List[DataPathFrame]:
    """Gets the data paths and frames from the list of files provided.

    Args:
      files: The files to be compressed.

    Returns:
        A list of paths and frames.

    Example:
        If we have an MRC file shape 1,2,3 at /tmp/0.mrc then::

          data = get_data_paths_and_frames("/tmp/0.mrc")

        data will be:
          [("/tmp/0.mrc",0), ("/tmp/0.mrc",1),("/tmp/0.mrc",2)]
    """
    data_paths = []
    for file in files:
        assert Path(file).is_file(), f"{file} is not a file."
        frames = np.max(mrcfile.mmap(file).data.shape)
        for frame in range(frames):
            data_paths.append((file, frame))
    return data_paths

# Data Loading

In [ ]:
class MRCFileDataset(Dataset):
    def __init__(
        self,
        dataset: List[str],
        transform: typing.Optional[torchvision.transforms.Compose] = None,
    ):
        self.dataset = dataset
       
        self.data_file =  mrcfile.mmap(dataset[0][0], mode="r")
        self.transform = transform

    def __len__(self) -> int:
        """Get the length of the dataset.
        Returns:
            int: Length of the dataset
        """
        return len(self.dataset)

    def __getitem__(
        self, idx: typing.Union[int, slice]
    ) -> np.ndarray[typing.Literal[4]]:
        """Get a sample from the dataset2 at idx position.
        Args:
            idx: idx to get the sample.
        Return:
            Frame with Shape: Height,Width,Channel
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()
        frame = decode_mrc_data_path(self.data_file, self.dataset[idx])
        if self.transform:
            frame = self.transform(frame)
        return frame

## Data Transforms

In [ ]:
def validate_chunk(chunk):
    # Assert valid chunk size
    if isinstance(chunk, int):
        assert chunk > 3
        ch, cw = (chunk,) * 2
    elif isinstance(chunk, tuple):
        ch, cw = chunk
        assert ch > 3
        assert cw > 3
    else:
        raise AssertionError("chunk must be int or tuple(int, int)")
    return ch, cw


class RandomChunk(object):
    def __init__(self, chunk: Union[int, tuple]):
        self.chunk_height, self.chunk_width = validate_chunk(chunk)

    def __call__(self, frame: np.ndarray) -> np.ndarray:
        """
        Args:
            frame: frame of data to take chunks out of. frame Shape ["Height", "Width", "Channels"]
        Returns:
        """
        y = np.random.randint(0, frame.shape[0] - self.chunk_height)
        x = np.random.randint(0, frame.shape[1] - self.chunk_width)
        chunk_frame = frame[y : y + self.chunk_height, x : x + self.chunk_width, :]
        return chunk_frame


class ExtractLevelFromHighres(object):
    """Extract Downsampled Low-Resolution batch from a high-resolution batch."""

    def __init__(self, level: int):
        """Initializes a new ExtractLevelFromHighre
        Args:

            level: level to downsample: results in a skip size of 2**level, i.e. an image that is 128x128 with level=2,3,4 the skip size is 4,8,16, resulting image sizes of 32x32,16x16,8x8 respectively
        """
        assert level >= 0

        self.level = level
        self.skip = 2**level

    def __call__(self, frame: np.ndarray) -> np.ndarray:
        """Batch to downsample from a high-resolution image.
        Args:
            frame: High resolution batch to downsample. Shape [height, width, channel]
        Return:
            np.ndarray: Downsampled batch: Shape [height, width, channel]
        """
        highres_frame = frame
        # Downsample by skip sampling
        lowres_frame = highres_frame[:: self.skip, :: self.skip]
        ph, pw = (np.shape(lowres_frame)[0] + 1) % 2, (
            np.shape(lowres_frame)[1] + 1
        ) % 2
        lowres_frame = np.pad(lowres_frame, ((0, ph), (0, pw), (0, 0)), mode="reflect")

        return lowres_frame


class LowresAndTargetsFromHighres(object):
    """Extract Low-Resolution and Targets from a high-resolution."""

    def __init__(self, padding: int, bit_depth: int):
        assert padding >= 0
        self.padding = padding
        self.bit_depth = bit_depth

    def __call__(self, highres) -> dict:
        """Downsample from a high-resolution.

        Args: highres: high-resolution images to downsample in the form [height, width, channels]
        """

        # Downsample by skip sampling
        lowres = highres[::2, ::2, :]

        # Pad the 2 spatial dimensions Height and Width
        lowres = np.pad(
            lowres,
            (
                (self.padding, self.padding),
                (self.padding, self.padding),
                (0, 0),
            ),
            mode="symmetric",
        )

        # Slice out each value of the pluses
        lmap = highres[1::2, :-1:2, :]
        rmap = highres[1::2, 2::2, :]
        umap = highres[:-1:2, 1::2, :]
        dmap = highres[2::2, 1::2, :]
        cmap = highres[1::2, 1::2, :]

        # Stack the vectors LRUDC order with dim [H,W,5,...]
        targets = np.stack([lmap, rmap, umap, dmap, cmap], axis=2)
        lowres[0] = lowres[0].astype(np.float32) / np.float32(2**self.bit_depth)
        targets[0] = targets[0].astype(np.float32) / np.float32(2**self.bit_depth)
        return dict(lowres=lowres, targets=targets)


class RandomChunkDataset(object):
    def __init__(
        self,
        padding,
        chunk_size: Optional[Tuple[int, ...]] = None,
        number_of_chunks: int = 1,
        bit_depth: int = 16,
        levels: int = 0,
    ):
        assert padding >= 0
        assert levels >= 0

        self.padding = padding
        self.chunk_size = chunk_size
        self.number_of_chunks = number_of_chunks
        self.bit_depth = bit_depth
        self.levels = levels

    def __call__(self, batch):
        ds = batch
        ds = ExtractLevelFromHighres(self.levels)(ds)

        # Generate Random Chunks
        if self.chunk_size:
            ds = RandomChunk(self.chunk_size)(ds)

        return LowresAndTargetsFromHighres(self.padding, self.bit_depth)(ds)

## Numpy DataLoader 

In [ ]:
def numpy_collate(batch):
    return tree_map(np.asarray, data.default_collate(batch))


class NumpyLoader(DataLoader):
    def __init__(
        self,
        dataset,
        batch_size=1,
        shuffle=False,
        sampler=None,
        batch_sampler=None,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
        timeout=0,
        worker_init_fn=None,
    ):
        super(self.__class__, self).__init__(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            sampler=sampler,
            batch_sampler=batch_sampler,
            num_workers=num_workers,
            collate_fn=numpy_collate,
            pin_memory=pin_memory,
            drop_last=drop_last,
            timeout=timeout,
            worker_init_fn=worker_init_fn,
        )

# Train Model

## Config Management 

Set the configuration for WandB and run settings

Change as needed

In [ ]:
config["learning_rate"] = 1e-5
config["epochs"] = 5
config["dataset_name"] = "name"
config["seed"] = 0
config["padding"] = 1
config["batch_size"] = 64
config["levels"] = 1
config["test_split"] = 0.2
config["model_name"] = "unet"
files = ["path_to_mrc_file"]
checkpoint_path = "absolute/checkpoint/path"
project_name="Project Name"
np.random.seed(config["seed"])

## Setup WandB
To login get the api key here: https://wandb.ai/authorize

In [ ]:
wandb.login()

In [ ]:
wandb.init(
    # set the wandb project where this run will be logged
    project=project_name,
    # track hyperparameters and run metadata
    config=config,
)

## Handle Data

This section deals with getting the data ready for training and testing

In [ ]:
data_paths_and_frames = get_data_paths_and_frames(files)

In [ ]:
train_data, test_data = model_selection.train_test_split(
    data_paths_and_frames, test_size=config["test_split"], random_state=config["seed"]
)

In [ ]:
print(len(train_data))
print(len(test_data))

In [ ]:
data_transforms = transforms.Compose([RandomChunkDataset(config["padding"])])

In [ ]:
# Train dataset gets transformed for the creation of maps
train_dataset = MRCFileDataset(train_data, data_transforms)
# Test dataset does not need transformation as this is used to evaluate the performance of the model so only needs highres images
test_data = MRCFileDataset(train_data)

In [ ]:
# Pin the memory to the GPU, this will mean the first epoch is slow but then will significantly faster afterward.
train_dataloader = NumpyLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True,
)
test_dataloader = NumpyLoader(
    test_data,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [ ]:
# Log at the end of every epoch
log_freq = len(train_dataloader)

## Ready up Kompressor

### Checkpointing

In [ ]:
options = ocp.CheckpointManagerOptions()
mngr = ocp.CheckpointManager(
    ocp.test_utils.erase_and_create_empty(checkpoint_path),
    options=options,
)

### Encoding

In [ ]:
encode_fn = kom.mapping.uint16.encode_values
decode_fn = kom.mapping.uint16.decode_values

### Kompressor

In [ ]:
compressor = FlaxKompressor(
    encode_fn=encode_fn,
    decode_fn=decode_fn,
    padding=config["padding"],
    model_fn=unet(config["neighbourhood"]),
    predictions_fn=RegressionPredictor,
).init(train_dataloader, seed = config["seed"])

In [ ]:
callbacks = [
    MetricsCallback(
        chunk=None,
        ds_train=None,
        ds_test=test_dataloader,
        log_freq=log_freq,
        levels=config["levels"],
    )
]

# Train

This section trains the model

In [ ]:
compressor.fit(
    ds_train=train_dataloader,
    start_step=0,
    end_step=config["epochs"],
    checkpoint_manager=mngr,
    callbacks=callbacks,
)

# Evalaute Performance

## Baseline Run Length Encoding

In [ ]:
baseline_data = MRCFileDataset(data_paths_and_frames)
baseline_dataloader = NumpyLoader(
    baseline_data,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

In [ ]:
def baseline(dataset, mode):
    summaries = defaultdict(list)
    writer_path = os.path.join(
        f"baseline/{mode}/run_length",
    )
    for highres in tqdm(dataset):
        summaries[f"{writer_path}"].extend(
            list(kom.image.metrics.mean_run_length(highres).flatten())
        )
    return jnp.array(summaries[writer_path]).flatten().mean()

In [ ]:
baseline(baseline_dataloader, "test")

# Load and Eval Saved Model

In [ ]:
def eval_compressor(dataset, levels, compressor, include_lowres=False):
    summaries = defaultdict(list)
    for highres in tqdm(dataset):
        lowres, level_encoded_maps = compressor.encode(
            highres, levels=levels, chunk=None, debug=True
        )

        for level, (
            level_highres,
            (level_encoded_maps, _),
            level_lowres,
        ) in enumerate(level_encoded_maps):
            level_highres_maps = kom.image.maps_from_highres(level_highres)

            writer_path = os.path.join(
                f"eval/level={level}/",
                f'lowres={"x".join(map(str, level_lowres.shape[1:]))}',
            )

            for label in level_encoded_maps.keys() & level_highres_maps.keys():
                summaries[f"{writer_path}/{label}/run_length"].extend(
                    list(
                        kom.image.metrics.mean_run_length(
                            level_encoded_maps[label]
                        ).flatten()
                    )
                )
            if include_lowres:
                summaries["lowres/run_length"].extend(
                    list(kom.image.metrics.mean_run_length(lowres).flatten())
                )
    run_lengths = []
    for key in summaries.keys():
        print(key)
        print(jnp.array(summaries[key]).flatten().mean())
        run_lengths.append(jnp.array(summaries[key]).flatten().mean())
    return jnp.array(run_lengths).flatten().mean()

In [ ]:
compressor = FlaxKompressor(
    encode_fn=encode_fn,
    decode_fn=decode_fn,
    padding=config["padding"],
    model_fn=unet(config["neighbourhood"]),
    predictions_fn=RegressionPredictor,
)

In [ ]:
checkpoint_epoch = 9

In [ ]:
compressor.avg_params = mngr.restore(checkpoint_epoch)["model"]["params"]

In [ ]:
eval_compressor(baseline_dataloader, 1, compressor)

# Verify Lossless Compression

In [ ]:
original_highres = jnp.array(next(iter(baseline_dataloader))[:1])
levels = 1
encoded_lowres, encoded_maps = compressor.encode(
    original_highres, levels=levels, debug=True
)
reconstructed_highres = compressor.decode(encoded_lowres, encoded_maps, debug=True)

fig, ax = plt.subplots(1, 2, figsize=(8, 4), facecolor="w")

plt.suptitle(f"lossless = {np.allclose(original_highres[0], reconstructed_highres[0])}")

plt.sca(ax[0])
plt.title("original")
plt.imshow(original_highres[0], cmap="RdBu")

plt.sca(ax[1])
plt.title("reconstructed")
plt.imshow(reconstructed_highres[0], cmap="RdBu")

plt.show()